In [1]:
%config Completer.use_jedi = False

In [2]:
# load module
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorboard as tb 
import pytorch_lightning as pl
from itertools import product
from sklearn import metrics


from pathlib import Path
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet , EncoderNormalizer , GroupNormalizer
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
from IPython.core.display import HTML

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'
#plt.rcParams['font.sans-serif'] = ['NanumGothic.ttf', 'sans-serif']

from my_funs_nei import data_processing , moving_average_alpha_both
pl.seed_everything(42)

/home/nplab/.local/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: /home/nplab/.local/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIlEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  warn(f"Failed to load image Python extension: {e}")
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will n

42

In [3]:
def get_training(data, transformation , max_prediction_length = 24 , max_encoder_length = 24*7):
    # traing data 생성
    training_cutoff = data["time_idx"].max() - max_prediction_length
    training = TimeSeriesDataSet(
        data[lambda x : x.time_idx <= training_cutoff],
        time_idx = "time_idx",
        target = "count",
        group_ids = ['h_dong'],
        min_encoder_length=max_encoder_length // 2,  # keep encoder length long (as it is in the validation set)
        max_encoder_length=max_encoder_length,
        min_prediction_length=1,
        max_prediction_length=24,
        static_categoricals=["h_dong"],
        time_varying_known_categoricals=["HOD", "DOW" , 'isHoliday', 'MOY'],
        time_varying_known_reals=['pops'],
        time_varying_unknown_categoricals=['precip_form'],
        time_varying_unknown_reals=['count','windspd' , 'temp' ,'precip', 'nei1'], 
        target_normalizer=GroupNormalizer(
            groups=["h_dong"], 
            transformation="relu",
            center = False
        ),
        add_relative_time_idx=True,
        add_target_scales=False,
        add_encoder_length=True,
        #allow_missing=True,
        allow_missing_timesteps = True,
        #predict_mode = False
        )
    return training


def TFTmodel(factor, epoch ,lr , hidden_size , atten_head , transformation , batch_size ):
    dong = '강남동'
    train_data = data_processing('nei_train.csv' , factor, moving_average_alpha_both)
    training = get_training(train_data, transformation, 24, 24*7)

    dataloader = training.to_dataloader(batch_size)
    validation = TimeSeriesDataSet.from_dataset(training, train_data, predict=True, stop_randomization=True)
    train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size*100, num_workers=0)

    early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
    lr_logger = LearningRateMonitor()  
    logger = TensorBoardLogger('ckpt')
    trainer = pl.Trainer(
        max_epochs=epoch,
        gpus=1,
        gradient_clip_val=0.1,
        limit_train_batches=20,  
        callbacks=[lr_logger],
        logger=logger,
    )

    tft = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=lr,
        hidden_size=hidden_size,  # most important hyperparameter apart from learning rate
        # number of attention heads. Set to up to 4 for large datasets
        attention_head_size=atten_head,
        dropout=0.1,  # between 0.1 and 0.3 are good values
        hidden_continuous_size=hidden_size,  # set to <= hidden_size
        output_size=7,  # 7 quantiles by default
        loss=QuantileLoss(),
        # reduce learning rate if no improvement in validation loss after x epochs
        reduce_on_plateau_patience=4,
    )

    trainer.fit( tft, 
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader)


    print(trainer.checkpoint_callback.best_model_path)
    
    ###### f2 score cal
    ######## params
    n = 3
    round_cut = 0.5
    
    prediction_df = pd.DataFrame()
    for file in sorted(os.listdir('./test_data'))[1:]:
        val_data = data_processing('./test_data/' + file , factor , moving_average_alpha_both)
        val_data = val_data.fillna(0)
        validation = TimeSeriesDataSet.from_dataset(training, val_data, predict=True, stop_randomization=True)
        val_dataloader = validation.to_dataloader(train=False, batch_size=16, num_workers=0)
        pred , x, idx_df = tft.predict(val_dataloader, mode='raw', return_x = True , return_index = True)
        idx = idx_df[idx_df['h_dong'] == dong].index[0]
        
        xyz = pd.DataFrame()
        delta_h = val_data['REG_DTIME'].unique()[1] - val_data['REG_DTIME'].unique()[0] 
        s = val_data['REG_DTIME'].unique().max() + delta_h
        e = val_data['REG_DTIME'].unique().max() + 24*delta_h

        #print(pd.date_range(s,e ,freq = 'h'))
        xyz.index = pd.date_range(s,e ,freq = 'h')
        
        xyz['prediction'] = pred['prediction'][idx, : , n]
        prediction_df = pd.concat([prediction_df, xyz])

    predition = prediction_df['prediction'][:-24]
    df_index = prediction_df.index[:-24]

    org_data = data_processing('nei_test.csv' , 0 , None)
    org_data = org_data[org_data['h_dong'] == dong]
    org_count = org_data[org_data['REG_DTIME'].isin(df_index)]['count']

    
    org_count = org_count.to_numpy()
    org_count = np.logical_not(org_count < 1)
    predition = predition
    predition = np.logical_not(predition < round_cut)
    return metrics.fbeta_score(org_count,predition , 2)

In [4]:
def objective(trail):
    
    param = {
        'epoch' : trail.suggest_categorical('epoch' , [100, 300, 500, 700]),
        'lr' : trail.suggest_categorical('lr',[0.5, 0.1, 0.05 , 0.01 , 0.005]),
        'factor' : trail.suggest_categorical('factor',[0.1, 0.2 , 0.3 ,0.4]),
        'batch_size' : trail.suggest_categorical('batch_size' , [4, 8, 16, 24, 32, 48,64]),
        'hidden_size' : trail.suggest_categorical('hidden_size' , [4, 8, 16, 24, 32, 48, 64]), 
        'atten_head' : trail.suggest_categorical('atten_head' , [2, 4, 6, 8, 12, 16]) , 
        'transformation': trail.suggest_categorical('transformation',['log' , 'logp1' , 'logit' , 'count' , 'softplus' , 'relu'])
    }
    
    return TFTmodel(**param)

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(lambda trial : objective(trial), n_trials=500)

[I 2023-01-02 16:38:13,781] A new study created in memory with name: no-name-2fb550ca-155c-4909-bae3-272d69810c6a
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: ckpt/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 538   
3  | prescalers                         | ModuleDict                      | 512   
4  | static_variable_selection          | VariableSelectionNetwork        | 4.6 K 
5  | encoder_variable_selection         | VariableSelectionNe

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [ ]:
fig = optuna.visualization.plot_parallel_coordinate(study)
fig.show()

In [ ]:
study.trials_dataframe().to_csv('optuna_result.csv' , index=False)